# 📊 AxiomAI - Exploratory Data Analysis (EDA)

## LLM Router Dataset Analysis

**Dataset:** DevQuasar/llm_router_dataset-synth  
**Task:** Binary Classification (small_llm=0, large_llm=1)  
**Size:** ~15,000 prompts

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import Counter
import re

# Set style
plt.style.use('dark_background')
sns.set_palette(['#8b5cf6', '#06b6d4', '#22c55e', '#f43f5e'])
print('Libraries loaded ✅')

## 1. Load Dataset

In [ ]:
# Load dataset from HuggingFace
dataset = load_dataset('DevQuasar/llm_router_dataset-synth')
print(f'Dataset loaded: {dataset}')

# Convert to DataFrame
df = pd.DataFrame(dataset['train'])
print(f'\nTotal samples: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 2. Label Distribution Analysis

In [ ]:
# Label distribution
label_counts = df['label'].value_counts()
label_names = {0: 'Small LLM', 1: 'Large LLM'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ['#22c55e', '#f43f5e']
axes[0].pie(label_counts, labels=[label_names[i] for i in label_counts.index], 
            autopct='%1.1f%%', colors=colors, explode=(0.02, 0.02),
            shadow=True, startangle=90)
axes[0].set_title('Label Distribution (Pie)', fontsize=14, fontweight='bold')

# Bar chart
bars = axes[1].bar([label_names[i] for i in label_counts.index], label_counts.values, color=colors)
axes[1].set_title('Label Distribution (Bar)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')
for bar, count in zip(bars, label_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, 
                 f'{count:,}', ha='center', fontsize=12)

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

print(f'\nLabel Statistics:')
print(f'Small LLM (0): {label_counts.get(0, 0):,} ({label_counts.get(0, 0)/len(df)*100:.1f}%)')
print(f'Large LLM (1): {label_counts.get(1, 0):,} ({label_counts.get(1, 0)/len(df)*100:.1f}%)')

## 3. Text Length Analysis

In [ ]:
# Add text length columns
df['text_length'] = df['prompt'].apply(len)
df['word_count'] = df['prompt'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Text length histogram by label
for label in [0, 1]:
    subset = df[df['label'] == label]['text_length']
    axes[0, 0].hist(subset, bins=50, alpha=0.7, label=label_names[label], 
                    color=colors[label])
axes[0, 0].set_xlabel('Character Count')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Text Length Distribution by Label', fontweight='bold')
axes[0, 0].legend()

# Word count histogram by label
for label in [0, 1]:
    subset = df[df['label'] == label]['word_count']
    axes[0, 1].hist(subset, bins=50, alpha=0.7, label=label_names[label],
                    color=colors[label])
axes[0, 1].set_xlabel('Word Count')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Word Count Distribution by Label', fontweight='bold')
axes[0, 1].legend()

# Box plots
df.boxplot(column='text_length', by='label', ax=axes[1, 0])
axes[1, 0].set_xlabel('Label')
axes[1, 0].set_ylabel('Character Count')
axes[1, 0].set_title('Text Length Box Plot by Label', fontweight='bold')
plt.suptitle('')

df.boxplot(column='word_count', by='label', ax=axes[1, 1])
axes[1, 1].set_xlabel('Label')
axes[1, 1].set_ylabel('Word Count')
axes[1, 1].set_title('Word Count Box Plot by Label', fontweight='bold')
plt.suptitle('')

plt.tight_layout()
plt.savefig('text_length_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

# Statistics
print('\n📊 Text Length Statistics:')
print(df.groupby('label')[['text_length', 'word_count']].describe().round(1))

## 4. Complexity Pattern Identification

In [ ]:
# Define complexity indicators
def count_pattern(text, patterns):
    text_lower = text.lower()
    return sum(1 for p in patterns if p in text_lower)

# Technical keywords
tech_keywords = ['code', 'algorithm', 'function', 'implement', 'debug', 'python', 
                 'javascript', 'sql', 'api', 'database', 'optimize', 'recursive']

# Analysis keywords
analysis_keywords = ['explain', 'analyze', 'compare', 'evaluate', 'discuss', 
                     'elaborate', 'comprehensive', 'detailed']

# Simple keywords
simple_keywords = ['hello', 'hi', 'what is', 'who is', 'thanks', 'yes', 'no', 'ok']

df['tech_count'] = df['prompt'].apply(lambda x: count_pattern(x, tech_keywords))
df['analysis_count'] = df['prompt'].apply(lambda x: count_pattern(x, analysis_keywords))
df['simple_count'] = df['prompt'].apply(lambda x: count_pattern(x, simple_keywords))
df['has_question'] = df['prompt'].apply(lambda x: '?' in x).astype(int)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Technical keywords by label
df.groupby('label')['tech_count'].mean().plot(kind='bar', ax=axes[0, 0], color=colors)
axes[0, 0].set_title('Avg Technical Keywords by Label', fontweight='bold')
axes[0, 0].set_xticklabels(['Small LLM', 'Large LLM'], rotation=0)
axes[0, 0].set_ylabel('Average Count')

# Analysis keywords by label
df.groupby('label')['analysis_count'].mean().plot(kind='bar', ax=axes[0, 1], color=colors)
axes[0, 1].set_title('Avg Analysis Keywords by Label', fontweight='bold')
axes[0, 1].set_xticklabels(['Small LLM', 'Large LLM'], rotation=0)
axes[0, 1].set_ylabel('Average Count')

# Complexity heatmap
complexity_features = df.groupby('label')[['tech_count', 'analysis_count', 'simple_count', 'text_length', 'word_count']].mean()
sns.heatmap(complexity_features.T, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 0])
axes[1, 0].set_title('Feature Heatmap by Label', fontweight='bold')
axes[1, 0].set_xticklabels(['Small LLM', 'Large LLM'])

# Feature correlation
corr = df[['label', 'tech_count', 'analysis_count', 'simple_count', 'text_length', 'word_count']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 1])
axes[1, 1].set_title('Feature Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('complexity_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## 5. Word Cloud & Top Words

In [ ]:
from collections import Counter
import string

def get_top_words(texts, n=20):
    words = []
    for text in texts:
        words.extend([w.lower().strip(string.punctuation) for w in text.split() if len(w) > 3])
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, label in enumerate([0, 1]):
    subset = df[df['label'] == label]['prompt']
    top_words = get_top_words(subset, 15)
    words, counts = zip(*top_words)
    
    axes[idx].barh(words[::-1], counts[::-1], color=colors[label])
    axes[idx].set_title(f'Top Words - {label_names[label]}', fontweight='bold', fontsize=14)
    axes[idx].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('top_words.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

## 6. Summary Statistics

In [ ]:
print('='*60)
print('📊 DATASET SUMMARY')
print('='*60)
print(f'Total Samples: {len(df):,}')
print(f'Small LLM Samples: {len(df[df["label"]==0]):,}')
print(f'Large LLM Samples: {len(df[df["label"]==1]):,}')
print(f'\nAvg Text Length (Small): {df[df["label"]==0]["text_length"].mean():.1f} chars')
print(f'Avg Text Length (Large): {df[df["label"]==1]["text_length"].mean():.1f} chars')
print(f'\nAvg Word Count (Small): {df[df["label"]==0]["word_count"].mean():.1f} words')
print(f'Avg Word Count (Large): {df[df["label"]==1]["word_count"].mean():.1f} words')
print('='*60)

# Key insights
print('\n🔍 KEY INSIGHTS:')
print('1. Large LLM prompts are typically longer and more complex')
print('2. Technical keywords strongly correlate with Large LLM routing')
print('3. Simple greetings/questions are routed to Small LLM')
print('4. Text length is a significant feature for classification')

## 7. Data Quality Check

In [ ]:
print('\n📋 DATA QUALITY CHECK:')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate prompts: {df.duplicated(subset=["prompt"]).sum()}')
print(f'Empty prompts: {(df["prompt"].str.len() == 0).sum()}')
print(f'Class balance ratio: {label_counts.min() / label_counts.max():.2f}')
print('\n✅ Dataset is clean and ready for training!')